In [1]:
import numpy as np
import pandas as pd
import itertools
from scipy.optimize import root

VOC = 38    
ISC = 8    
Ns = 54     

def extract_villalva_parameters(Voc_n, Isc_n, Ns, n=1.3):
    q, k, T_n = 1.60e-19, 1.38e-23, 298.15
    Vt = (Ns * k * T_n) / q
    Rp = (Voc_n / 0.05) / Isc_n
    Rs = 0.01 * (Voc_n / Isc_n)
    I0_n = (Isc_n - (Voc_n - Isc_n * Rs) / Rp) / (np.exp(Voc_n / (n * Vt)) - 1.0)
    return [Voc_n, Isc_n, Rs, Rp, I0_n, Vt]

pv_params = extract_villalva_parameters(VOC, ISC, Ns)

def solve_module_current_with_bypass(V, G, pv_params, n=1.3):
    Voc_n, Isc_n, Rs, Rp, I0_n, Vt = pv_params
    Ipv = (G / 1000.0) * Isc_n if G > 1.0 else 0.0
    
    I_guess = Ipv
    for _ in range(8):
        vd_over_avt = (V + I_guess * Rs) / (n * Vt)
        vd_over_avt = min(50.0, max(-50.0, vd_over_avt)) 
        expr = np.exp(vd_over_avt)
        f = Ipv - I0_n * (expr - 1.0) - (V + I_guess * Rs) / Rp - I_guess
        df = -I0_n * (Rs / (n * Vt)) * expr - (Rs / Rp) - 1.0
        I_guess -= f / df
        
    I0_bd = 1e-6
    Vt_bd = 0.026 
    v_bd_ratio = -V / Vt_bd
    v_bd_ratio = min(50.0, max(-50.0, v_bd_ratio))
    I_bypass = I0_bd * (np.exp(v_bd_ratio) - 1.0)
    return I_guess + I_bypass

def solve_mna_network(V_array, shading_matrix, switches, pv_params, initial_guess):
    flat_G = np.array(shading_matrix).flatten()

    def residual(variables):
        v10, v11, v12 = variables[0], variables[1], variables[2]
        v20, v21, v22 = variables[3], variables[4], variables[5]
        i_s1a, i_s1b  = variables[6], variables[7]
        i_s2a, i_s2b  = variables[8], variables[9]

        I0 = solve_module_current_with_bypass(V_array - v10, flat_G[0], pv_params)
        I1 = solve_module_current_with_bypass(V_array - v11, flat_G[1], pv_params)
        I2 = solve_module_current_with_bypass(V_array - v12, flat_G[2], pv_params)
        I3 = solve_module_current_with_bypass(v10 - v20, flat_G[3], pv_params)
        I4 = solve_module_current_with_bypass(v11 - v21, flat_G[4], pv_params)
        I5 = solve_module_current_with_bypass(v12 - v22, flat_G[5], pv_params)
        I6 = solve_module_current_with_bypass(v20 - 0, flat_G[6], pv_params)
        I7 = solve_module_current_with_bypass(v21 - 0, flat_G[7], pv_params)
        I8 = solve_module_current_with_bypass(v22 - 0, flat_G[8], pv_params)

        res = np.zeros(10)
        res[0] = I0 - I3 - i_s1a             
        res[1] = I1 - I4 + i_s1a - i_s1b     
        res[2] = I2 - I5 + i_s1b             
        res[3] = I3 - I6 - i_s2a             
        res[4] = I4 - I7 + i_s2a - i_s2b     
        res[5] = I5 - I8 + i_s2b             
        res[6] = v10 - v11 if switches['S1_a'] else i_s1a
        res[7] = v11 - v12 if switches['S1_b'] else i_s1b
        res[8] = v20 - v21 if switches['S2_a'] else i_s2a
        res[9] = v21 - v22 if switches['S2_b'] else i_s2b
        return res

    sol = root(residual, initial_guess, method='hybr')
    v10_f, v11_f, v12_f = sol.x[0], sol.x[1], sol.x[2]
    total_current = (solve_module_current_with_bypass(V_array - v10_f, flat_G[0], pv_params) +
                     solve_module_current_with_bypass(V_array - v11_f, flat_G[1], pv_params) +
                     solve_module_current_with_bypass(V_array - v12_f, flat_G[2], pv_params))
    return max(0.0, total_current), list(sol.x)

def generate_dataset(num_samples):
    switch_names = ['S1_a', 'S1_b', 'S2_a', 'S2_b']
    all_combos = list(itertools.product([True, False], repeat=4))
    dataset = []
    np.random.seed(42)
    
    v_sweep_internal = np.linspace(1.0, 3 * VOC, 40)
    
    landmark_voltages = [5.0, 15.0, 30.0, 45.0, 60.0, 75.0, 90.0, 100.0]

    for sample in range(num_samples):
        if (sample + 1) % 10 == 0 or sample == 0:
            print(f"Analyzing shading matrix scenario {sample + 1}/{num_samples}...")
            
        shading = np.random.choice(np.arange(100, 1100, 100), size=(3, 3))
        flat_shading = shading.flatten()
        
        best_power = -1
        best_config_idx = -1
        for idx, combo in enumerate(all_combos):
            switches = dict(zip(switch_names, combo))
            guess_config = [2*v_sweep_internal[0]/3]*3 + [v_sweep_internal[0]/3]*3 + [0.0]*4
            
            max_p = 0
            for v in v_sweep_internal:
                i_out, guess_config = solve_mna_network(v, shading, switches, pv_params, guess_config)
                p_out = v * i_out
                if p_out > max_p: max_p = p_out
            if max_p > best_power:
                best_power = max_p
                best_config_idx = idx
                
        baseline_switches = {'S1_a': True, 'S1_b': True, 'S2_a': True, 'S2_b': True}
        guess = [2*landmark_voltages[0]/3]*3 + [landmark_voltages[0]/3]*3 + [0.0]*4
        
        for v_terminal in landmark_voltages:
            i_out, guess = solve_mna_network(v_terminal, shading, baseline_switches, pv_params, guess)
            p_terminal = v_terminal * i_out
            
            row_data = {
                'Output_Voltage': v_terminal,
                'Output_Power': p_terminal,
                'Optimal_Config_Label': best_config_idx
            }
            for i in range(9):
                row_data[f'Mod_{i}_G'] = flat_shading[i]
                
            dataset.append(row_data)
        
    return pd.DataFrame(dataset)

if __name__ == "__main__":
    print("Starting high-accuracy physical tracking engine...")
    df = generate_dataset(num_samples=500)
    
    try:
        df.to_csv("pv_shading_dataset.csv", index=False)
        print(f"\nSuccess! Built dataset with {len(df)} structured landmark rows.")
    except PermissionError:
        df.to_csv("pv_shading_dataset_new.csv", index=False)
        print(f"\nSaved successfully to fallback: 'pv_shading_dataset_new.csv'")

Starting high-accuracy physical tracking engine...
Analyzing shading matrix scenario 1/500...
Analyzing shading matrix scenario 10/500...
Analyzing shading matrix scenario 20/500...
Analyzing shading matrix scenario 30/500...
Analyzing shading matrix scenario 40/500...
Analyzing shading matrix scenario 50/500...
Analyzing shading matrix scenario 60/500...
Analyzing shading matrix scenario 70/500...
Analyzing shading matrix scenario 80/500...
Analyzing shading matrix scenario 90/500...
Analyzing shading matrix scenario 100/500...
Analyzing shading matrix scenario 110/500...
Analyzing shading matrix scenario 120/500...
Analyzing shading matrix scenario 130/500...
Analyzing shading matrix scenario 140/500...
Analyzing shading matrix scenario 150/500...
Analyzing shading matrix scenario 160/500...
Analyzing shading matrix scenario 170/500...
Analyzing shading matrix scenario 180/500...
Analyzing shading matrix scenario 190/500...
Analyzing shading matrix scenario 200/500...
Analyzing shadi